# CO₂: separar producción y transferencia con una química común

Química fija **theta_nv2**, sin reajustar biomasa, YAN, glucosa, fructosa o etanol. Se cruzan dos producciones y dos leyes de transferencia; los modelos y sus parámetros guardados permanecen intactos. Los coeficientes del diagnóstico sólo existen en memoria y en las tablas del EXECUTED.

| Código | Producción | Transferencia |
|---|---|---|
| A | Histórica, con sus modulaciones | Histórica |
| B | Histórica, con sus modulaciones | Nueva |
| C | Ligada directamente a crecimiento y etanol | Histórica |
| D | Ligada directamente a crecimiento y etanol | Nueva |

**Transferencia:** comparar B−A y D−C. **Producción:** comparar C−A y D−B. En cada comparación se reajustan los mismos dos coeficientes de transferencia/observación, con el mismo objetivo. Es una comparación controlada de formulaciones, no una identificación causal de mecanismos biológicos.

**Particiones de CO₂:** calibrar LAB019–021 y evaluar LAB016–018; ejecutar también la inversa. Ninguna muestra de holdout entra al optimizador. La línea experimental une exactamente las medianas horarias de la RAW convertida, no la señal GUI filtrada.

La producción histórica requiere el contrafactual sin nutrición, calculado con el mismo theta_nv2. Se conserva el conjunto histórico de activación, O₂ y nutrición sólo para diagnosticar su efecto; no se incorpora a los modelos nuevos. Aquí no se comparan de nuevo transportadores frente a activación directa por YAN: ambos modelos nuevos comparten la capa de gas que se está aislando.


In [ ]:
from pathlib import Path
import sys,json,hashlib,importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from IPython.display import display,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/sources.csv').exists())
sys.dont_write_bytecode=True
sys.path.insert(0,str(ROOT/'src'))
assert Path(sys.prefix).name.lower()=='fermentation'
from wine_fermentation.comparison import observations,partition,PARTITIONS,LABS
from wine_fermentation.models.historical.calibration import load
from wine_fermentation.shared.solubility import saturation
from dataclasses import replace

def protected_hashes():
    return {str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
            for folder in ('src','configs','results') for p in (ROOT/folder).rglob('*')
            if p.is_file() and '__pycache__' not in p.parts and p.suffix!='.pyc'}
protected_before=protected_hashes()
raw_table=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',
    usecols=['lab','time_h','record_kind','remaining_volume_L','co2_raw_g_l_h'])
obs_all=observations(raw_table)
V=load(ROOT)
theta=V.thetas['theta_nv2'].copy()
# Normalize identical dosing by the remaining volume, without adding nutrient volume.
volumes={}
for lab in LABS:
    events=raw_table[(raw_table.lab==lab)&(raw_table.record_kind=='withdrawal_after')].sort_values('time_h')
    et=events.time_h.to_numpy();ev=events.remaining_volume_L.to_numpy()
    def volume(t,et=et,ev=ev):
        return np.r_[2.,ev][np.searchsorted(et,np.asarray(t,float),side='right')]
    volumes[lab]=volume
    if lab in LABS[3:]:
        batch=V.batches[lab]
        V.batches[lab]=replace(batch,pulses={**batch.pulses,
            'N':tuple((t,d*2/float(volume(t))) for t,d in batch.pulses['N'])})
V.prepare('theta_nv2',theta,LABS)
print('Química theta_nv2 preparada una sola vez; producción histórica y nueva usan las mismas trayectorias con nutrición.')


## Condiciones compartidas y ecuaciones del diagnóstico

Ambas leyes reciben la misma saturación provisional:
\[
C_{sat}=1.69\exp[-0.032(T-20)]\exp(0.0016E)\exp[-0.0012(G+F)].
\]
Su escala se fija en 1 en el ensayo principal. Se aplica Cd(0)=0 g/L en los seis LAB y las cuatro combinaciones, por instrucción del usuario. Las mediciones del carbodoseur no se usan como iniciales en este ensayo.

La producción nueva es la que usan actualmente las propuestas:
\[
q_{CO_2,prod}=\left(\frac{\mu}{12.72}+\frac{\beta_G+\beta_F}{1.05}\right)X.
\]
La histórica se toma del cálculo histórico existente, con theta_nv2 y sus modulaciones fijas; no se reajusta su respuesta a nutrición ni O₂.

Transferencia histórica:
\[
q_{CO_2,gas}=k_{CO_2,release}\,C_d\left(0.05+0.95\frac{C_d}{C_d+C_{sat}}\right).
\]
Transferencia nueva:
\[
q_{CO_2,gas}=k_La\max(C_d-C_{sat},0).
\]
En ambas: \(dC_d/dt=q_{CO_2,prod}-q_{CO_2,gas}\) y predicción del sensor = ganancia × caudal físico. Los dos coeficientes de transferencia tienen unidades h⁻¹, pero pertenecen a leyes distintas: sus valores no deben interpretarse como magnitudes físicas intercambiables.

El volumen remanente se comparte. Una retirada homogénea conserva Cd y reduce el inventario V·Cd; no añade volumen la nutrición. Se comprueban balances de concentración y masa, incluyendo el CO₂ retirado en muestras.

A es una **combinación histórica bajo condiciones comunes**; su Cd inicial, saturación e inicialización del ajuste difieren de la referencia histórica guardada. No es una reproducción de esa referencia. La correlación de solubilidad continúa siendo provisional.


La rejilla común fusiona coincidencias numéricas separadas por menos de 1e-9 h al unir tiempos. Evita dividir por pasos casi nulos; no suaviza los datos ni cambia los modelos. Se registran los nodos fusionados y se verifica también la ley constitutiva, además del balance.

**Fin de curva compartido:** se usan medianas RAW de diez minutos sólo para detectar el final: 2 h consecutivas ≤0.02 g/(L·h), con cobertura suficiente y sin actividad posterior ≥0.05. El corte incluye siempre la última muestra offline. Si no hay confirmación antes de acabar el registro, se marca censura derecha, no fin confirmado. Para ajustar y dibujar se mantienen las medianas horarias. No se fuerzan a cero observaciones ni estados.

La corrección de concentración de las nutriciones por volumen remanente se conserva en este diagnóstico; su forma de salto y sus horarios no cambian.

In [ ]:
def merge_grid(values):
    values=np.unique(np.asarray(values,float))
    # Merge floating-point coincidences, preserving the latest timestamp of each group.
    return values[np.r_[np.diff(values)>1e-9,True]]

common={};context_rows=[]
for lab in LABS:
    batch=V.batches[lab];core=V.cores['theta_nv2',lab];cache=V.cache['theta_nv2',lab]
    base_t=core.index.to_numpy(float)
    withdrawn=raw_table[(raw_table.lab==lab)&(raw_table.record_kind=='withdrawal_after')].time_h.to_numpy()
    nutrient=np.array([t for t,d in batch.pulses['N']],float)
    extra=np.r_[withdrawn,nutrient];extra=extra[(extra>=base_t[0])&(extra<=base_t[-1])]
    unmerged_grid=np.unique(np.r_[base_t,np.arange(base_t[0],base_t[-1],.1),extra])
    grid=merge_grid(unmerged_grid)
    rates=[]
    for ti,row in core.iterrows():
        q=V.base.kinetic_terms(theta,V.base.temperature_at(batch,ti),row.X,row.N,row.G,row.F,row.E)
        rates.append((q['mu']/12.72+(q['beta_g']+q['beta_f'])/1.05)*row.X)
    states={s:np.interp(grid,base_t,core[s]) for s in ('X','N','G','F','E')}
    T=np.interp(grid,batch.time,batch.temperature_c)
    sat=saturation(T,states['E'],states['G'],states['F'])
    cd0=0.
    volume=volumes[lab](grid)
    assert np.all(np.diff(volume)<=1e-12)
    common[lab]=dict(time=grid,dt=np.diff(grid),sat=sat,Cd0=cd0,V=volume,
        sources={'H':np.interp(grid,cache['time'],cache['source']),
                 'N':np.interp(grid,base_t,np.asarray(rates))},states=states)
    context_rows.append(dict(lab=lab,theta='theta_nv2',V0_L=volume[0],Vfinal_L=volume[-1],
        merged_roundoff_nodes=len(unmerged_grid)-len(grid),min_step_h=float(np.diff(grid).min()),
        Cd0_g_l=cd0,Csat0_g_l=sat[0],Cd0_status='cero asumido por instrucción',
        nutrient_pulses=str(batch.pulses['N'])))
LOD=.05
QUIET=.02
END_WINDOW_H=2.
END_BIN_H=1./6.

def cutoff_from_medians(ten,last_offline,available):
    if last_offline>available+1e-8:raise ValueError('El registro no cubre la última muestra offline')
    active=ten[ten>=LOD]
    last_active_end=float((active.index.max()+1)*END_BIN_H) if len(active) else 0.
    nwindow=int(round(END_WINDOW_H/END_BIN_H))
    for i in range(len(ten)-nwindow+1):
        window=ten.iloc[i:i+nwindow]
        start=float(window.index[0]*END_BIN_H);end=float((window.index[-1]+1)*END_BIN_H)
        if start+1e-9<last_active_end or end+1e-9<last_offline:continue
        if np.all(np.diff(window.index)==1) and window.le(QUIET).all():
            return dict(cut_h=min(end,available),right_censored=False,last_offline_h=last_offline,
                        last_detectable_end_h=last_active_end,quiet_window_start_h=start,quiet_window_end_h=end)
    return dict(cut_h=available,right_censored=True,last_offline_h=last_offline,
                last_detectable_end_h=last_active_end,quiet_window_start_h=np.nan,quiet_window_end_h=np.nan)

# Independent policy checks: final offline sample, later activity, gaps, and censoring.
_test=pd.Series(0.,index=np.arange(48));_test.iloc[:6]=.1
_check=cutoff_from_medians(_test,3.,8.)
assert _check['cut_h']>=3 and not _check['right_censored']
_late=_test.copy();_late.iloc[30]=.1
_check=cutoff_from_medians(_late,3.,8.)
assert _check['quiet_window_start_h']>=(31/6)-1e-9 and _check['cut_h']>=(31/6)+2-1e-9
_gap=pd.Series(0.,index=np.r_[np.arange(5),np.arange(10,48)])
_check=cutoff_from_medians(_gap,0.,8.)
assert _check['quiet_window_start_h']>=10/6-1e-9
assert cutoff_from_medians(pd.Series(.1,index=np.arange(48)),3.,8.)['right_censored']

horizons={};horizon_rows=[]
for lab in LABS:
    available=float(common[lab]['time'][-1])
    last_offline=0.
    for table in (V.chem,V.density,V.biomass):
        points=table[table.lab.eq(lab)]
        if 'is_NUT' in points:points=points[~points.is_NUT]
        if len(points):last_offline=max(last_offline,float(points.t_h.max()))
    native=raw_table[(raw_table.lab==lab)&(raw_table.record_kind=='native')&raw_table.time_h.between(0.,available)].dropna(subset=['co2_raw_g_l_h']).copy()
    dt=float(np.median(np.diff(np.unique(native.time_h))))
    native['bin']=np.floor(native.time_h/END_BIN_H+1e-10).astype(int)
    ten=native.groupby('bin').agg(lo=('time_h','min'),hi=('time_h','max'),n=('time_h','size'),y=('co2_raw_g_l_h','median'))
    complete=(ten.hi-ten.lo>=.75*END_BIN_H)&(ten.n>=.75*END_BIN_H/dt)&((ten.index+1)*END_BIN_H<=available+1e-9)
    ten=ten.loc[complete,'y']
    check=cutoff_from_medians(ten,last_offline,available)
    horizons[lab]=check['cut_h'];horizon_rows.append(dict(lab=lab,available_h=available,**check))
    assert horizons[lab]>=last_offline-1e-8
obs={lab:obs_all[lab][obs_all[lab].time_h.between(common[lab]['time'][0],horizons[lab])].copy() for lab in LABS}
assert all(len(o)>0 for o in obs.values())
display(pd.DataFrame(horizon_rows))
print('Fin: 2 h consecutivas ≤0.02 g/(L·h), sin reactivación posterior ≥0.05 y cubriendo la última muestra offline. Sin confirmación: censurado, no terminado.')

display(pd.DataFrame(context_rows))
COMBOS={'A':('H','H'),'B':('H','N'),'C':('N','H'),'D':('N','N')}
LABELS={'A':'A · producción histórica / transferencia histórica',
        'B':'B · producción histórica / transferencia nueva',
        'C':'C · producción nueva / transferencia histórica',
        'D':'D · producción nueva / transferencia nueva'}
COLORS={'A':'#0072B2','B':'#CC79A7','C':'#E69F00','D':'#009E73'}

def reservoir(lab,production,transfer,k,scale=1.,refine=None):
    c=common[lab];t=c['time'];source=c['sources'][production];sat=c['sat']*scale
    if refine is not None:
        grid=merge_grid(np.r_[t,np.arange(t[0],t[-1],refine)])
        source=np.interp(grid,t,source);sat=np.interp(grid,t,sat);t=grid
    dt=np.diff(t);cd=np.zeros(len(t));cd[0]=c['Cd0'];qgas=np.zeros(len(t))
    if transfer=='H':qgas[0]=k*cd[0]*(.05+.95*cd[0]/(cd[0]+sat[0]))
    else:qgas[0]=k*max(cd[0]-sat[0],0.)
    for i,step in enumerate(dt,1):
        inventory=cd[i-1]+step*source[i-1]
        if transfer=='N':
            cd[i]=inventory if inventory<=sat[i] else (inventory+step*k*sat[i])/(1+step*k)
        else:
            aa=1+step*k;b=sat[i]*(1+.05*step*k)-inventory
            root=np.sqrt(b*b+4*aa*inventory*sat[i])
            cd[i]=2*inventory*sat[i]/(root+b) if b>=0 else (root-b)/(2*aa)
        qgas[i]=(inventory-cd[i])/step
    expected=k*cd*(.05+.95*cd/(cd+sat)) if transfer=='H' else k*np.maximum(cd-sat,0.)
    assert np.max(abs(qgas-expected))<1e-7,'La integración no respeta la ley de transferencia'
    residual=np.diff(cd)-dt*(source[:-1]-qgas[1:])
    assert np.max(abs(residual))<1e-9 and np.min(cd)>=-1e-12
    volume=volumes[lab](t)
    produced_g=float(np.sum(volume[:-1]*dt*source[:-1]))
    emitted_g=float(np.sum(volume[:-1]*dt*qgas[1:]))
    sampled_g=float(np.sum(np.maximum(-np.diff(volume),0)*cd[1:]))
    mass_error=volume[-1]*cd[-1]-volume[0]*cd[0]-produced_g+emitted_g+sampled_g
    assert abs(mass_error)<1e-8
    f=pd.DataFrame(dict(qprod=source,qgas=qgas,Cd=cd,Csat=sat,V=volume,Md=volume*cd),index=t)
    f.attrs.update(mass_error_g=float(mass_error),concentration_error=float(np.max(abs(residual))),
        produced_g=produced_g,emitted_g=emitted_g,sampled_g=sampled_g)
    return f

# Sanity checks independent of the observed signal.
for transfer in ('H','N'):
    for lab in LABS:reservoir(lab,'N',transfer,1.)
print('Balances de concentración y masa correctos en ambas leyes, incluyendo las retiradas.')


## Ajustes controlados: sólo transferencia y ganancia

Objetivo = suma de la pérdida cuadrática media con umbral de detección por fermentación; todos los LAB pesan igual. Si la observación es ≥0.05 g/(L·h), se utiliza predicción−observación. Si es inferior, el error es cero hasta que la predicción supera 0.05, y después es predicción−0.05. No se filtra la predicción. RMSE y MSE ordinarios se muestran aparte como métricas descriptivas. Se optimiza la ganancia mediante la derivada monótona de la pérdida con umbral para cada coeficiente de transferencia y se explora su perfil en una malla logarítmica, seguido de refinamiento local. Se comprueban también los extremos.

Cotas comunes: coeficiente de transferencia 0.01–20 h⁻¹ y ganancia 0.05–3. La escala de saturación permanece fija. Estas cotas son controles numéricos del ensayo, no una validación física. Los valores 50 y 100 h⁻¹ se calculan sólo para visualizar la meseta fuera de cota; no se seleccionan como ajustes.


In [ ]:
def signal_error(predicted,observed):
    predicted=np.asarray(predicted,float);observed=np.asarray(observed,float)
    return np.where(observed>=LOD,predicted-observed,np.maximum(predicted-LOD,0.))

# Same below-detection treatment as the original notebook: no penalty up to LOD.
np.testing.assert_allclose(signal_error([.03,.08,.08],[.01,.01,.10]),[0.,.03,-.02])

def optimal_gain(xy):
    x=np.concatenate([x for x,y in xy]);y=np.concatenate([y for x,y in xy])
    weights=np.concatenate([np.full(len(x),1./len(x)) for x,y in xy])
    def derivative(g):return float(np.dot(weights*x,signal_error(g*x,y)))
    lo,hi=.05,3.
    if derivative(lo)>=0:gain=lo
    elif derivative(hi)<=0:gain=hi
    else:
        for _ in range(55):
            mid=(lo+hi)/2
            if derivative(mid)>0:hi=mid
            else:lo=mid
        gain=(lo+hi)/2
    error=signal_error(gain*x,y)
    return float(gain),float(np.dot(weights,error**2))

_fixture=[(np.array([.01,.03,.1]),np.array([0.,.02,.2]))]
_g,_=optimal_gain(_fixture)
_control=minimize_scalar(lambda g:np.mean(signal_error(g*_fixture[0][0],_fixture[0][1])**2),bounds=(.05,3.),method='bounded',options={'xatol':1e-12})
assert abs(_g-_control.x)<1e-6

KGRID=np.geomspace(.01,20.,41)
fits={};curves={};profile_rows=[];metric_rows=[];mass_rows=[]
fit_calls=[]

def fit_case(combo,training,scale=1.,keep_profile=False):
    production,transfer=COMBOS[combo];train,holdout=partition(training)
    evaluated={}
    def objective(logk):
        k=float(np.exp(logk));key=float(logk)
        if key in evaluated:return evaluated[key]
        xy=[]
        for lab in train:
            f=reservoir(lab,production,transfer,k,scale)
            o=obs[lab]
            xy.append((np.interp(o.time_h,f.index,f.qgas),o.co2_raw_g_l_h.to_numpy()))
            fit_calls.append((training,lab))
        gain,cost=optimal_gain(xy)
        evaluated[key]=(cost,gain);return cost,gain
    sampled=[objective(np.log(k))[0] for k in KGRID]
    best_index=int(np.argmin(sampled))
    left=np.log(KGRID[max(0,best_index-1)]);right=np.log(KGRID[min(len(KGRID)-1,best_index+1)])
    sol=minimize_scalar(lambda z:objective(z)[0],bounds=(left,right),method='bounded',options={'xatol':1e-7})
    assert sol.success
    candidates=[(sampled[i],np.log(KGRID[i])) for i in range(len(KGRID))]+[(float(sol.fun),float(sol.x))]
    _,best_log=min(candidates);cost,gain=objective(best_log);k=float(np.exp(best_log))
    fit=dict(combination=combo,training=training,production=production,transfer=transfer,
             transfer_h_inv=k,gain=gain,objective=cost,scale=scale,success=bool(sol.success),
             active_bound='superior' if k>=20*(1-1e-5) else 'inferior' if k<=.01*(1+1e-5) else '',
             training_labs=train,holdout_labs=holdout,n_profile_points=len(evaluated))
    if keep_profile:
        for kval in np.r_[KGRID,50.,100.]:
            value,g=objective(np.log(kval))
            profile_rows.append(dict(combination=combo,training=training,k=kval,objective=value,gain=g))
    return fit

for training in PARTITIONS:
    for combo in COMBOS:
        f=fit_case(combo,training,keep_profile=True);fits[training,combo]=f
        for lab in LABS:
            frame=reservoir(lab,*COMBOS[combo],f['transfer_h_inv'])
            frame['qpred']=f['gain']*frame.qgas;curves[training,combo,lab]=frame
            o=obs[lab];pred=f['gain']*np.interp(o.time_h,frame.index,frame.qgas)
            error=pred-o.co2_raw_g_l_h.to_numpy()
            role='Calibración CO₂' if lab in f['training_labs'] else 'Holdout CO₂'
            for window,mask in [('Completo',np.ones(len(o),bool)),('0–24 h',o.time_h.to_numpy()<24),
                ('24–90 h',(o.time_h.to_numpy()>=24)&(o.time_h.to_numpy()<90)),('≥90 h',o.time_h.to_numpy()>=90)]:
                if not mask.any():continue
                e=error[mask]
                metric_rows.append(dict(training=training,combination=combo,lab=lab,role=role,window=window,n=int(mask.sum()),
                    LOD_MSE=float(np.mean(signal_error(pred[mask],o.co2_raw_g_l_h.to_numpy()[mask])**2)),
                    MSE=float(np.mean(e**2)),RMSE=float(np.sqrt(np.mean(e**2))),MAE=float(np.mean(abs(e))),bias=float(e.mean()),
                    observed_hourly_peak_h=float(o.time_h.iloc[np.argmax(o.co2_raw_g_l_h)]),
                    predicted_hourly_peak_h=float(o.time_h.iloc[np.argmax(pred)])))
            mass_rows.append(dict(training=training,combination=combo,lab=lab,**frame.attrs))
    print('Completada partición',training)
assert all(lab in partition(training)[0] for training,lab in fit_calls)
fit_table=pd.DataFrame(fits.values());metrics=pd.DataFrame(metric_rows);profiles=pd.DataFrame(profile_rows)
mass_checks=pd.DataFrame(mass_rows)
full=metrics[metrics.window=='Completo']
summary=full.groupby(['training','combination','role'])[['RMSE','MAE','MSE','LOD_MSE','bias']].mean()
display(fit_table[['training','combination','transfer_h_inv','gain','objective','active_bound','success']])
display(Markdown('**Media por fermentación. RMSE/MSE ordinarios son descriptivos; LOD_MSE corresponde al criterio del ajuste.** Comparar combinaciones dentro de la misma partición y rol; los holdouts de las dos particiones contienen distintos LAB.'))
display(summary)
display(full[['training','combination','lab','role','RMSE','bias','observed_hourly_peak_h','predicted_hourly_peak_h']])
assert mass_checks.mass_error_g.abs().max()<1e-8
for (training,combo),f in fits.items():
    actual=full[(full.training==training)&(full.combination==combo)&(full.role=='Calibración CO₂')].LOD_MSE.sum()
    assert np.isclose(actual,f['objective'],rtol=1e-10,atol=1e-12)
print('Holdout excluido del ajuste. Objetivos reproducidos y balances verificados para los 48 casos.')


## Producción calculada antes del depósito
Estas curvas son producciones internas, no mediciones del sensor. Las dos producciones utilizan exactamente las mismas trayectorias químicas con nutrición; la histórica añade su contrafactual y sus modulaciones.

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(15,9))
fig.subplots_adjust(top=.80,bottom=.08,hspace=.43,wspace=.27)
fig.suptitle('Producción de CO₂ calculada · química común theta_nv2',y=.99,fontsize=15)
for ax,lab in zip(axes.flat,LABS):
    c=common[lab]
    ax.plot(c['time'][c['time']<=horizons[lab]],c['sources']['H'][c['time']<=horizons[lab]],label='Producción histórica con modulaciones',color='#0072B2')
    ax.plot(c['time'][c['time']<=horizons[lab]],c['sources']['N'][c['time']<=horizons[lab]],label='Producción ligada a crecimiento y etanol',color='#D55E00')
    ax.set_title(lab);ax.set_xlim(0,horizons[lab]);ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('CO₂ producido [g/(L·h)]');ax.grid(alpha=.25)
h,l=axes[0,0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.92),ncol=2,title='Fuentes calculadas · antes de transferencia y ganancia')
plt.show()


## CO₂ emitido frente a las medianas usadas para ajustar
Fila superior: calibración; fila inferior: holdout. La ganancia de observación está incluida en las predicciones. La línea experimental une las medianas y conserva la cobertura del objetivo. El eje Y se ajusta automáticamente para mostrar completas las curvas.

Con Cd(0)=0, el caudal inicial es cero en las cuatro combinaciones. Durante la fermentación la ley histórica puede emitir por debajo de Csat; la nueva requiere superarlo. Las predicciones se muestran sin filtrar y el eje Y es automático.

In [ ]:
for training in PARTITIONS:
    train,holdout=partition(training)
    fig,axes=plt.subplots(2,3,figsize=(16,9.5))
    fig.subplots_adjust(top=.74,bottom=.08,hspace=.43,wspace=.27)
    fig.suptitle('CO₂ emitido · química común theta_nv2\nCalibración: '+', '.join(train)+' | Holdout: '+', '.join(holdout),y=.99,fontsize=14)
    for ax,lab in zip(axes.flat,train+holdout):
        o=obs[lab];ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.5,zorder=5,label='Medianas horarias del sensor')
        for combo in COMBOS:
            f=curves[training,combo,lab];keep=f.index<=horizons[lab];ax.plot(f.index[keep],f.qpred[keep],color=COLORS[combo],label=LABELS[combo])
        ax.set_title(lab+' · '+('Calibración CO₂' if lab in train else 'Holdout CO₂'),fontsize=11)
        ax.set_xlim(0,horizons[lab])
        ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('CO₂ emitido [g/(L·h)]');ax.set_ylim(bottom=0);ax.margins(y=.06);ax.grid(alpha=.25)
    h,l=axes[0,0].get_legend_handles_labels()
    fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.90),ncol=2,title='Datos y combinaciones · producción / transferencia',fontsize=10)
    plt.show()


## theta_nv2 con los parámetros de CO₂ del bloque antiguo

Esta figura mantiene **theta_nv2, producción histórica y transferencia histórica**. Compara el conjunto compartido del bloque antiguo «mismos coeficientes de capa, sólo cambia theta» con el ajuste A de LAB019–021. Los valores antiguos se leen a precisión completa de configs/co2_reference_parameters.csv: fila theta_full, train_019_021, Z2, reference_20_45, offline_30min, modo v2. Esa fila fue el conjunto de gas compartido entre los tres theta en la figura antigua; no es el Z2 específico de theta_nv2.

Se muestran cuatro curvas: todos los coeficientes antiguos; cambiar sólo la ganancia; cambiar sólo la liberación; A actual con sus coeficientes. El cambio aislado de escala de saturación se cuantifica en una tabla, porque aquí 0.999955 y 1 son prácticamente iguales y sus líneas se superpondrían.

**Condiciones comunes del diagnóstico:** Cd inicial cero, volumen remanente, pulsos de concentración corregida, medianas horarias y política de final compartida. **Ninguna predicción se filtra.** Por tanto esta figura aísla los coeficientes de gas; no pretende reproducir el procesamiento ni todas las condiciones originales del notebook archivado. Este bloque no optimiza ni guarda parámetros adicionales.


In [ ]:
old_reference=pd.read_csv(ROOT/'configs/co2_reference_parameters.csv',float_precision='round_trip')
selected=old_reference[(old_reference.temperature=='reference_20_45')&
    (old_reference.filtering=='offline_30min')&(old_reference['mode']=='v2')&
    (old_reference.theta=='theta_full')&(old_reference.training=='train_019_021')&
    (old_reference.stage=='Z2')]
assert len(selected)==1,'La referencia antigua debe ser única'
old_row=selected.iloc[0]
old_gas={key:float(old_row[key]) for key in ('kCO2_release_h','CO2sat_scale','matrix_gain')}
np.testing.assert_allclose(list(old_gas.values()),[2.000835,.999955,1.054708],rtol=0,atol=1e-6)
current_A=fits['train_019_021','A']
coefficient_scenarios={
    'Antiguos':dict(kCO2_release_h=old_gas['kCO2_release_h'],CO2sat_scale=old_gas['CO2sat_scale'],matrix_gain=old_gas['matrix_gain']),
    'Sólo ganancia':dict(kCO2_release_h=old_gas['kCO2_release_h'],CO2sat_scale=old_gas['CO2sat_scale'],matrix_gain=current_A['gain']),
    'Sólo liberación':dict(kCO2_release_h=current_A['transfer_h_inv'],CO2sat_scale=old_gas['CO2sat_scale'],matrix_gain=old_gas['matrix_gain']),
    'A actual':dict(kCO2_release_h=current_A['transfer_h_inv'],CO2sat_scale=1.,matrix_gain=current_A['gain']),
    'Sólo escala = 1':dict(kCO2_release_h=old_gas['kCO2_release_h'],CO2sat_scale=1.,matrix_gain=old_gas['matrix_gain'])}
display(Markdown('**Parámetros utilizados (sin reajuste en este bloque):**'))
display(pd.DataFrame(coefficient_scenarios).T.rename_axis('Curva'))
coefficient_curves={};coefficient_metrics=[];scale_effect=[]
for lab in LABS:
    assert common[lab]['Cd0']==0.
    for scenario,parameters in coefficient_scenarios.items():
        frame=reservoir(lab,'H','H',parameters['kCO2_release_h'],scale=parameters['CO2sat_scale'])
        frame['qpred']=parameters['matrix_gain']*frame.qgas
        coefficient_curves[scenario,lab]=frame
        o=obs[lab];prediction=np.interp(o.time_h,frame.index,frame.qpred)
        error=prediction-o.co2_raw_g_l_h.to_numpy()
        coefficient_metrics.append(dict(lab=lab,scenario=scenario,
            role='Calibración de A' if lab in current_A['training_labs'] else 'Holdout de A',
            RMSE=float(np.sqrt(np.mean(error**2))),
            LOD_MSE=float(np.mean(signal_error(prediction,o.co2_raw_g_l_h.to_numpy())**2))))
    np.testing.assert_allclose(coefficient_curves['A actual',lab].qpred,curves['train_019_021','A',lab].qpred,rtol=0,atol=1e-12)
    np.testing.assert_allclose(coefficient_curves['Sólo ganancia',lab].qpred,
        coefficient_curves['Antiguos',lab].qpred*current_A['gain']/old_gas['matrix_gain'],rtol=1e-12,atol=1e-12)
    mask=coefficient_curves['Antiguos',lab].index<=horizons[lab]
    difference=coefficient_curves['Sólo escala = 1',lab].qpred-coefficient_curves['Antiguos',lab].qpred
    scale_effect.append(dict(lab=lab,max_absolute_scale_effect_g_l_h=float(np.max(abs(difference[mask])))))
coefficient_metrics=pd.DataFrame(coefficient_metrics)
plot_labels={
    'Antiguos':'theta_nv2 · parámetros CO₂ antiguos',
    'Sólo ganancia':'theta_nv2 · cambiar sólo ganancia',
    'Sólo liberación':'theta_nv2 · cambiar sólo liberación',
    'A actual':'theta_nv2 · A actual (coeficientes reajustados)'}
plot_colors={'Antiguos':'#0072B2','Sólo ganancia':'#009E73','Sólo liberación':'#CC79A7','A actual':'#D55E00'}
train,holdout=partition('train_019_021')
fig,axes=plt.subplots(2,3,figsize=(16,9.5))
fig.subplots_adjust(top=.74,bottom=.08,hspace=.43,wspace=.27)
fig.suptitle('CO₂ con theta_nv2 · efecto de los parámetros de la capa\nProducción y transferencia históricas · mismas condiciones del diagnóstico',y=.99,fontsize=14)
for ax,lab in zip(axes.flat,train+holdout):
    o=obs[lab];ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.5,label='Medianas horarias del sensor',zorder=5)
    for scenario in ('Antiguos','Sólo liberación','Sólo ganancia','A actual'):
        frame=coefficient_curves[scenario,lab];keep=frame.index<=horizons[lab]
        ax.plot(frame.index[keep],frame.qpred[keep],color=plot_colors[scenario],
            ls='--' if scenario=='A actual' else '-',lw=1.6,label=plot_labels[scenario])
    ax.set_title(lab+' · '+('Calibración de A' if lab in train else 'Holdout de A'),fontsize=11)
    ax.set_xlim(0,horizons[lab]);ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('CO₂ emitido [g/(L·h)]')
    ax.set_ylim(bottom=0);ax.margins(y=.06);ax.grid(alpha=.25)
h,l=axes[0,0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.90),ncol=2,
    title='Cambios aislados respecto al conjunto antiguo · sin filtrar la predicción',fontsize=10)
plt.show()
display(Markdown('**Comparación numérica sobre las mismas medianas y ventanas:** los valores de la referencia antigua se evalúan aquí con el objetivo del diagnóstico. No son los RMSE filtrados del notebook archivado.'))
display(coefficient_metrics.pivot(index=['role','lab'],columns='scenario',values='RMSE'))
display(Markdown('**Efecto aislado de cambiar la escala de saturación antigua a 1:**'))
display(pd.DataFrame(scale_effect))
print('Control: A reconstruida idéntica a A existente; cambio de ganancia verificado; ninguna optimización adicional.')


## Efectos separados, tras reajustar los mismos dos coeficientes
Diferencia de pérdida media con umbral (LOD_MSE) por LAB. Un valor negativo favorece la segunda combinación de cada comparación. No se mezclan los grupos de calibración y holdout ni se comparan medias de distintos LAB como si fueran el mismo conjunto.

In [ ]:
wide=full.groupby(['training','role','combination']).LOD_MSE.mean().unstack('combination')
effects=[]
for (training,role),row in wide.iterrows():
    for effect,first,second in [('Transferencia con producción histórica','A','B'),('Transferencia con producción nueva','C','D'),
                                ('Producción con transferencia histórica','A','C'),('Producción con transferencia nueva','B','D')]:
        effects.append(dict(training=training,role=role,effect=effect,comparison=second+' − '+first,
            delta_LOD_MSE=float(row[second]-row[first]),change_percent=float(100*(row[second]/row[first]-1))))
effects=pd.DataFrame(effects)
display(effects)
display(Markdown('**Error por tramos:** permite localizar si el cambio ayuda al inicio, en la zona activa o al final.'))
display(metrics[metrics.window!='Completo'].groupby(['training','combination','role','window']).RMSE.mean().unstack('window'))


## Perfil del coeficiente de transferencia
La ganancia se reajusta en cada punto usando sólo entrenamiento. El eje Y divide el objetivo por el mejor valor encontrado dentro de la cota: una curva plana cerca de 1 indica poca información sobre ese coeficiente. La línea vertical marca 20 h⁻¹; 50–100 h⁻¹ son exploraciones diagnósticas, no ajustes seleccionados.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,9))
fig.subplots_adjust(top=.79,bottom=.08,hspace=.36,wspace=.25)
fig.suptitle('Información del ajuste sobre el coeficiente de transferencia',y=.99,fontsize=15)
for ax,combo in zip(axes.flat,COMBOS):
    for training,color in [('train_019_021','#0072B2'),('train_016_018','#D55E00')]:
        p=profiles[(profiles.combination==combo)&(profiles.training==training)].sort_values('k')
        best=fits[training,combo]
        ax.plot(p.k,p.objective/best['objective'],color=color,label='Calibración '+('LAB019–021' if training=='train_019_021' else 'LAB016–018'))
        ax.scatter([best['transfer_h_inv']],[1.],color=color,s=25)
    ax.axvline(20,color='gray',ls=':');ax.set_xscale('log');ax.set_yscale('log')
    ax.set_title(LABELS[combo],fontsize=10);ax.set_xlabel('Coeficiente de transferencia [h⁻¹]');ax.set_ylabel('Objetivo / mínimo en cota');ax.grid(alpha=.25)
h,l=axes[0,0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.92),ncol=2,title='Datos que entran al perfil · puntos: ajuste elegido dentro de cota')
plt.show()
plateau=[]
for (training,combo),fit in fits.items():
    p=profiles[(profiles.training==training)&(profiles.combination==combo)].set_index('k')
    outside=p.loc[[50.,100.]].objective.min()
    plateau.append(dict(training=training,combination=combo,k_fit=fit['transfer_h_inv'],
        improvement_beyond_bound_percent=100*(fit['objective']-outside)/fit['objective']))
display(pd.DataFrame(plateau))


## CO₂ disuelto y transferencia: predicciones internas
Se muestran LAB019–021 bajo ambos ajustes. La misma curva de saturación y el mismo punto inicial se aplican a las cuatro combinaciones. Fila superior: Cd y saturación; fila inferior: emisión física antes de la ganancia. Estos caudales no son observaciones.

In [ ]:
for training in PARTITIONS:
    train,holdout=partition(training)
    fig,axes=plt.subplots(2,3,figsize=(16,10))
    fig.subplots_adjust(top=.70,bottom=.08,hspace=.43,wspace=.28)
    role='Calibración CO₂' if 'LAB019' in train else 'Holdout CO₂'
    fig.suptitle('Depósito disuelto y emisión física · '+role+' LAB019–021\nQuímica, saturación y Cd inicial comunes · antes de ganancia',y=.99,fontsize=14)
    for j,lab in enumerate(LABS[3:]):
        for combo in COMBOS:
            f=curves[training,combo,lab]
            keep=f.index<=horizons[lab]
            axes[0,j].plot(f.index[keep],f.Cd[keep],color=COLORS[combo],label=LABELS[combo])
            axes[1,j].plot(f.index[keep],f.qgas[keep],color=COLORS[combo])
        c=common[lab]
        keep=c['time']<=horizons[lab]
        axes[0,j].plot(c['time'][keep],c['sat'][keep],color='black',ls=':',label='Saturación provisional común')
        axes[0,j].scatter([0],[c['Cd0']],color='black',s=22,label='Cd inicial asumido: cero')
        axes[0,j].set_title(lab+' · Depósito disuelto');axes[1,j].set_title(lab+' · Emisión física')
        axes[0,j].set_ylabel('CO₂ disuelto [g/L]');axes[1,j].set_ylabel('CO₂ emitido [g/(L·h)]')
        for ax in axes[:,j]:ax.set_xlim(0,horizons[lab]);ax.set_xlabel('Tiempo desde inoculación [h]');ax.grid(alpha=.25)
    h,l=axes[0,0].get_legend_handles_labels()
    fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.90),ncol=2,title='Predicciones internas y condición inicial',fontsize=10)
    plt.show()

initial_emissions=pd.DataFrame([dict(training=training,combination=combo,lab=lab,
    qgas_initial_g_l_h=float(frame.qgas.iloc[0]),qpred_initial_g_l_h=float(frame.qpred.iloc[0]))
    for (training,combo,lab),frame in curves.items() if lab in LABS[3:]])
display(Markdown('**Condición inicial común:** Cd(0)=0 y emisión inicial nula en las cuatro combinaciones.'))
display(initial_emissions)


## Sensibilidad de la saturación, separada del ensayo principal
Se multiplican por 0.5, 1, 2 y 4 las mismas curvas de saturación en ambas leyes y se repiten los ajustes sólo de transferencia/ganancia. Esto explora compensación entre parámetros; no valida ninguna escala ni elige una usando el holdout. Estos resultados no sustituyen la comparación principal de escala 1.

In [ ]:
sensitivity=[]
for scale in (.5,1.,2.,4.):
    for training in PARTITIONS:
        train,holdout=partition(training)
        for combo in COMBOS:
            f=fits[training,combo] if scale==1. else fit_case(combo,training,scale=scale)
            for role,labs in [('Calibración CO₂',train),('Holdout CO₂',holdout)]:
                rmses=[];mses=[];lod_mses=[]
                for lab in labs:
                    frame=reservoir(lab,*COMBOS[combo],f['transfer_h_inv'],scale=scale)
                    o=obs[lab];e=f['gain']*np.interp(o.time_h,frame.index,frame.qgas)-o.co2_raw_g_l_h.to_numpy()
                    mses.append(float(np.mean(e**2)));rmses.append(float(np.sqrt(np.mean(e**2))))
                    lod_mses.append(float(np.mean(signal_error(e+o.co2_raw_g_l_h.to_numpy(),o.co2_raw_g_l_h.to_numpy())**2)))
                sensitivity.append(dict(scale=scale,training=training,combination=combo,role=role,
                    mean_RMSE=float(np.mean(rmses)),mean_MSE=float(np.mean(mses)),mean_LOD_MSE=float(np.mean(lod_mses)),k=f['transfer_h_inv'],gain=f['gain'],active_bound=f['active_bound']))
sensitivity=pd.DataFrame(sensitivity)
assert all(lab in partition(training)[0] for training,lab in fit_calls)
display(sensitivity[sensitivity.role=='Holdout CO₂'].pivot(index=['training','combination'],columns='scale',values='mean_RMSE'))
display(sensitivity[sensitivity.role=='Calibración CO₂'][['scale','training','combination','k','gain','active_bound']])
fig,axes=plt.subplots(1,2,figsize=(15,5))
fig.subplots_adjust(top=.65,bottom=.15,wspace=.25)
fig.suptitle('Sensibilidad a la saturación · error de CO₂ en holdout\nCoeficientes reajustados sólo con la calibración correspondiente',y=.99,fontsize=14)
for ax,training in zip(axes,PARTITIONS):
    for combo in COMBOS:
        s=sensitivity[(sensitivity.training==training)&(sensitivity.combination==combo)&(sensitivity.role=='Holdout CO₂')]
        ax.plot(s.scale,s.mean_RMSE,marker='o',color=COLORS[combo],label=LABELS[combo])
    ax.set_title('Calibración '+('LAB019–021 → holdout LAB016–018' if training=='train_019_021' else 'LAB016–018 → holdout LAB019–021'),fontsize=11)
    ax.set_xlabel('Multiplicador diagnóstico de saturación');ax.set_ylabel('Media RMSE por LAB [g/(L·h)]');ax.grid(alpha=.25)
h,l=axes[0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.85),ncol=2,title='Combinaciones · no seleccionar escala utilizando el holdout',fontsize=9)
plt.show()


## Onset manual: comparación sin reajustar coeficientes

Se reproduce la **convención de onset manual de pyomo-doe**, documentada en `co2_reformulation/02_nutrition_structure_comparison.source.ipynb`, celdas 15 y 19: el intervalo de referencia de la activación pasa a ser **[0, onset_manual]**. Se conservan las fracciones históricas de inicio y duración. El onset es una referencia para construir la activación gradual, no la hora exacta de cruce del umbral del sensor.

Horas manuales editables: LAB016=24.2491, LAB017=22.2981, LAB018=22.6475, LAB019=28, LAB020=32 y LAB021=24 h. Se utilizan como entradas diagnósticas, sin estimarlas ni recalibrarlas.

Para cada LAB se comparan dos activaciones con cada uno de los dos conjuntos de CO₂ ya disponibles: **parámetros antiguos compartidos** y **parámetros de A**. Dentro de cada pareja se mantienen química theta_nv2, producción histórica antes de la activación, O₂, respuesta a nutrición, temperatura, saturación, Cd inicial cero, volumen, coeficientes de transferencia y ganancia. Sólo cambia la activación; el depósito se vuelve a integrar. La nutrición conserva su hora y concentración actuales del diagnóstico.

No se desplaza toda la curva ni se filtra la predicción. Las curvas manuales son un **diagnóstico condicionado al onset indicado**, no una nueva validación independiente. Esta comparación no introduce otros términos de los notebooks antiguos ni exporta parámetros.


In [ ]:
# EDITAR AQUÍ para ensayar otras horas sin cambiar modelos ni configuraciones guardadas.
MANUAL_ONSET_H={'LAB016':24.2491,'LAB017':22.2981,'LAB018':22.6475,
                'LAB019':28.,'LAB020':32.,'LAB021':24.}
activation_start_fraction=float(V.old['chem_activation_start_fraction'])
activation_duration_fraction=float(V.old['chem_activation_duration_fraction'])

def manual_activation(time,onset_h):
    start=activation_start_fraction*onset_h
    duration=max(activation_duration_fraction*onset_h,.25)
    z=np.clip((np.asarray(time,float)-start)/duration,0.,1.)
    return z*z*(3.-2.*z)

# Boundary check of the original bounded smoothstep convention.
for value in MANUAL_ONSET_H.values():
    start=activation_start_fraction*value;end=start+max(activation_duration_fraction*value,.25)
    np.testing.assert_allclose(manual_activation([0,start,(start+end)/2,end],value),[0.,0.,.5,1.],atol=1e-14)
manual_sets={'Antiguos':coefficient_scenarios['Antiguos'].copy(),'A':coefficient_scenarios['A actual'].copy()}
manual_parameters_before=json.dumps(manual_sets,sort_keys=True)
manual_inputs_before={lab:{key:V.cache['theta_nv2',lab][key].copy()
    for key in ('source','q0','qn','ramp','multiplier','fferm','gate','sat_base')} for lab in LABS}
manual_gates=[];manual_curves={};manual_rows=[]
for lab in LABS:
    c=common[lab];cache=V.cache['theta_nv2',lab];onset=float(MANUAL_ONSET_H[lab])
    assert 0<onset<c['time'][-1]
    activity=1.+(float(V.old['pulse_activity_gain'])-1.)*cache['ramp']
    qbio=np.maximum(activity*cache['q0']+cache['ramp']*(cache['qn']-cache['q0']),0.)
    before_activation=cache['multiplier']*cache['fferm']*qbio
    np.testing.assert_allclose(cache['gate']*before_activation,cache['source'],rtol=1e-12,atol=1e-12)
    original_H=c['sources']['H'].copy()
    c['sources']['H_manual']=manual_activation(c['time'],onset)*np.interp(c['time'],cache['time'],before_activation)
    assert np.array_equal(original_H,c['sources']['H'])
    start=activation_start_fraction*onset;duration=max(activation_duration_fraction*onset,.25)
    manual_gates.append(dict(lab=lab,onset_manual_h=onset,activation_start_h=start,
        activation_complete_h=start+duration,nutrition_h=float(V.batches[lab].pulses['N'][0][0])))
    for param_set,parameters in manual_sets.items():
        reference=coefficient_curves['Antiguos' if param_set=='Antiguos' else 'A actual',lab]
        for gate_kind in ('Por azúcar','Onset manual'):
            if gate_kind=='Por azúcar':frame=reference
            else:
                frame=reservoir(lab,'H_manual','H',parameters['kCO2_release_h'],scale=parameters['CO2sat_scale'])
                frame['qpred']=parameters['matrix_gain']*frame.qgas
            assert float(frame.Cd.iloc[0])==0. and float(frame.qgas.iloc[0])==0.
            manual_curves[param_set,gate_kind,lab]=frame
            o=obs[lab];prediction=np.interp(o.time_h,frame.index,frame.qpred)
            pulse=V.batches[lab].pulses['N'][0][0]
            for window,mask in [('Completo',np.ones(len(o),bool)),('Antes de nutrición',o.time_h.to_numpy()<pulse),
                                ('Desde nutrición',o.time_h.to_numpy()>=pulse)]:
                if not mask.any():continue
                actual=o.co2_raw_g_l_h.to_numpy()[mask];pred=prediction[mask]
                manual_rows.append(dict(lab=lab,parameter_set=param_set,activation=gate_kind,window=window,n=int(mask.sum()),
                    RMSE=float(np.sqrt(np.mean((pred-actual)**2))),LOD_MSE=float(np.mean(signal_error(pred,actual)**2))))
manual_gate_table=pd.DataFrame(manual_gates);manual_metrics=pd.DataFrame(manual_rows)
display(Markdown('**Referencia manual, intervalo efectivo de activación y nutrición:**'))
display(manual_gate_table)
display(Markdown('**Coeficientes de cada pareja: permanecen fijos al cambiar la activación.**'))
display(pd.DataFrame(manual_sets).T)
styles={('Antiguos','Por azúcar'):('#0072B2','-','Parámetros antiguos · activación por azúcar'),
        ('Antiguos','Onset manual'):('#009E73','--','Parámetros antiguos · onset manual'),
        ('A','Por azúcar'):('#D55E00','-','Parámetros A · activación por azúcar'),
        ('A','Onset manual'):('#CC79A7','--','Parámetros A · onset manual')}
fig,axes=plt.subplots(2,3,figsize=(16,10.5))
fig.subplots_adjust(top=.68,bottom=.08,hspace=.43,wspace=.27)
fig.suptitle('CO₂ con onset manual · theta_nv2\nCada pareja conserva sus parámetros de CO₂; sólo cambia la activación',y=.99,fontsize=14)
for ax,lab in zip(axes.flat,LABS[3:]+LABS[:3]):
    o=obs[lab];ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.5,zorder=5,label='Medianas horarias del sensor')
    for (param_set,gate_kind),(color,linestyle,label) in styles.items():
        frame=manual_curves[param_set,gate_kind,lab];keep=frame.index<=horizons[lab]
        ax.plot(frame.index[keep],frame.qpred[keep],color=color,ls=linestyle,lw=1.6,label=label)
    ax.axvline(V.batches[lab].pulses['N'][0][0],color='#8E44AD',ls=':',lw=.9,label='Hora de nutrición')
    ax.axvline(MANUAL_ONSET_H[lab],color='gray',ls='-.',lw=.9,label='Referencia onset manual')
    ax.set_title(lab+' · onset manual '+format(MANUAL_ONSET_H[lab],'.2f')+' h',fontsize=11)
    ax.set_xlim(0,horizons[lab]);ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('CO₂ emitido [g/(L·h)]')
    ax.set_ylim(bottom=0);ax.margins(y=.06);ax.grid(alpha=.25)
h,l=axes[0,0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.90),ncol=2,
    title='Activaciones comparadas · diagnóstico condicionado, sin reajuste',fontsize=10)
plt.show()
display(Markdown('**Error sobre las mismas medianas y ventanas, con los coeficientes fijos:**'))
display(manual_metrics.pivot(index=['parameter_set','lab','window'],columns='activation',values='RMSE'))
assert manual_parameters_before==json.dumps(manual_sets,sort_keys=True)
for lab,fields in manual_inputs_before.items():
    for key,value in fields.items():assert np.array_equal(value,V.cache['theta_nv2',lab][key]),(lab,key)
print('Onset manual verificado: química, producción previa a activación, O₂, nutrición, saturación y coeficientes conservados; sólo cambia la activación y se reintegra el depósito.')


## Verificaciones y lectura final
Se comprueba independencia del holdout, conservación de masa y estabilidad de los resultados principales al refinar el paso de la capa de gas. El chequeo de paso conserva las fuentes y la saturación interpoladas: no vuelve a calibrar ni prueba por sí solo la precisión de toda la química.

In [ ]:
numerical=[]
for (training,combo),fit in fits.items():
    for step in (.1,.025):
        costs=[]
        for lab in fit['training_labs']:
            frame=curves[training,combo,lab] if step==.1 else reservoir(lab,*COMBOS[combo],fit['transfer_h_inv'],refine=step)
            o=obs[lab];e=signal_error(fit['gain']*np.interp(o.time_h,frame.index,frame.qgas),o.co2_raw_g_l_h.to_numpy())
            costs.append(float(np.mean(e**2)))
        numerical.append(dict(training=training,combination=combo,max_step_h=step,objective=sum(costs)))
numerical=pd.DataFrame(numerical)
display(numerical.pivot(index=['training','combination'],columns='max_step_h',values='objective'))
assert protected_before==protected_hashes(),'Se modificó un modelo o un parámetro guardado'
print('Modelos, configs y results intactos. Sin CSV, PNG ni parámetros exportados.')
display(Markdown('**Para interpretar:** primero comparar las cuatro combinaciones dentro de una misma partición, con escala de saturación 1. Después consultar los efectos pareados y los perfiles: una mejora numérica no demuestra que O₂, la saturación o la ley elegida tengan validez física. La sensibilidad de saturación se informa aparte.'))

for training in PARTITIONS:
    means=full[(full.training==training)&(full.role=='Holdout CO₂')].groupby('combination').RMSE.mean()
    best=means.idxmin()
    display(Markdown('**Holdout '+', '.join(partition(training)[1])+':** menor media de RMSE con '+LABELS[best]+', '+format(means[best],'.4f')+' g/(L·h). Comparación entre las cuatro combinaciones bajo condiciones comunes; no demuestra validez física.'))
